In [94]:
import pandas as pd
from pathlib import Path
import re
import numpy as np
import os
import json
import dbstep.Dbstep as db
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolAlign, rdDetermineBonds, rdmolops
from rdkit.Chem.rdchem import Mol
from scipy.spatial.transform import Rotation
h_to_eV = 27.2114
bohr_to_A = 0.529177

In [143]:
class OrcaReader:
    """ORCA results reader with unified atom index labeling.

    Atom handling:
    - atom_index_map: Dict[int, str] mapping 1-based atom index -> label string
    - set_atoms(list[int]) assigns numeric string labels automatically
    - set_atoms(dict[int,str]) assigns explicit labels
    - set_atom_labels(list[str]|dict[int,str]) updates labels

    Result keys follow: f"{prefix}_{label}".
    """
    def __init__(self, directory, basename="singlepoint", out_file_path=None, nbo_block_index=0, nbo_job_number=None):
        self.directory = Path(directory)
        self.basename = basename
        self.outfile = Path(out_file_path) if out_file_path else self.directory / f"{self.basename}.out"
        self.nbo_block_index = nbo_block_index
        self.nbo_job_number = nbo_job_number

        # Core data stores
        self.results = {}
        self.propertyjson = {}
        self.gbwjson = {}

        # Unified atom index mapping (1-based indices)
        self.atom_index_map = {}

        # NBO parsing caches (kept for future extension)
        self.nbo_sections = []
        self.nbo_section = None
        self._outfile_content = None
        self._nbo_blocks_meta = []
        self._job_headers = []
        self.nboresults = {}

    # ---------------- Atom management ----------------
    def set_atoms(self, atoms):
        """Define atom indices with optional labels.
        atoms: list[int] or dict[int,str] (1-based indices).
        list -> labels are their numeric string; dict -> explicit labels.
        """
        if isinstance(atoms, dict):
            self.atom_index_map = {int(k): (str(v) if v is not None else str(k)) for k, v in atoms.items()}
        elif isinstance(atoms, (list, tuple)):
            self.atom_index_map = {int(k): str(int(k)) for k in atoms}
        else:
            raise TypeError("atoms must be list[int] or dict[int,str]")

    def set_atom_labels(self, labels):
        """Update labels (dict[int,str] or list[str] matching current order)."""
        if not self.atom_index_map:
            print("Set atoms before labels.")
            return
        if isinstance(labels, dict):
            for k, v in labels.items():
                k = int(k)
                if k in self.atom_index_map and isinstance(v, str):
                    self.atom_index_map[k] = v.strip()
        elif isinstance(labels, (list, tuple)):
            if len(labels) != len(self.atom_index_map):
                print("Label list length must match number of atoms.")
                return
            for key, new_label in zip(self.atom_index_map.keys(), labels):
                if isinstance(new_label, str):
                    self.atom_index_map[key] = new_label.strip()
        else:
            raise TypeError("labels must be dict or list[str]")

    @property
    def atom_indices(self):
        return list(self.atom_index_map.keys())

    def _atom_key(self, prefix, atom_index):
        label = self.atom_index_map.get(atom_index, str(atom_index))
        return f"{prefix}_{label}"
    
    # -------------------- configuration helpers --------------------
    def set_outfile(self, out_file_path):
        changed = Path(out_file_path) != self.outfile
        self.outfile = Path(out_file_path)
        if changed:
            self.nbo_sections = []
            self.nbo_section = None
            self._outfile_content = None
            self._nbo_blocks_meta = []
            self._job_headers = []
            self._clear_nbo_data()

    def set_nbo_block_index(self, index:int):
        prev_index = self.nbo_block_index
        self.nbo_block_index = index
        if self.nbo_sections:
            if 0 <= index < len(self.nbo_sections):
                if index != prev_index:
                    self.nbo_section = self.nbo_sections[index]
                    self._clear_nbo_data()
            else:
                print(f"Requested NBO block index {index} out of range (found {len(self.nbo_sections)} blocks).")

    def set_nbo_job_number(self, job_number:int|None):
        prev_job = self.nbo_job_number
        self.nbo_job_number = job_number
        if prev_job != job_number:
            self.nbo_section = None
            self._clear_nbo_data()

    # -------------------- internal clear helper --------------------
    def _clear_nbo_data(self):
        """Clear previously parsed NBO results and any NBO charge entries in self.results."""
        nbo_keys = [k for k in list(self.results.keys()) if k.startswith("nbo_")]
        for k in nbo_keys:
            del self.results[k]
        self.nboresults = {}

    # ---------------- File parsing ----------------
    def parse_property_json(self):
        fname = f"{self.basename}.property.json"
        path = self.directory / fname
        if path.exists():
            try:
                self.propertyjson = json.load(open(path))
            except Exception as e:
                print(f"Failed to load {fname}: {e}")
                self.propertyjson = {}
        else:
            print(f"{fname} not found; skipping.")
            self.propertyjson = {}

    def parse_gbw_json(self):
        fname = f"{self.basename}.json"
        path = self.directory / fname
        if path.exists():
            try:
                self.gbwjson = json.load(open(path))
            except Exception as e:
                print(f"Failed to load {fname}: {e}")
                self.gbwjson = {}
        else:
            self.gbwjson = {}

    # ---------------- Nested helpers ----------------
    def _nested_get(self, obj, path_list):
        cur = obj
        for p in path_list:
            try:
                cur = cur[p]
            except Exception as e:
                print(e)
                return None
        return cur

    def _last_geometry(self):
        geoms = self._nested_get(self.propertyjson, ["Geometries"])
        if geoms and isinstance(geoms, list):
            return geoms[-1]
        return None

    # ---------------- Charge extraction ----------------
    def _charge_table(self, key):
        geom = self._last_geometry()
        return self._nested_get(geom, [key, 0, "AtomicCharges"]) if geom else None

    def _extract_charges(self, table_key, prefix, preserve_case=False, local_atom_index_map=None):
        """Generic extractor supporting alternate atom index maps.

        local_atom_index_map: optional dict[int,str]; if None uses self.atom_index_map.
        """
        if local_atom_index_map is None:
            local_atom_index_map = self.atom_index_map
        if not local_atom_index_map:
            print("No atoms defined for charge extraction.")
            return
        table = self._charge_table(table_key)
        if table is None:
            print(f"{table_key} charges not found; skipping.")
            return
        name_prefix = prefix if preserve_case else prefix.lower()
        for idx, label in local_atom_index_map.items():
            value = None
            entry = self._nested_get(table, [idx - 1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            elif entry:
                value = entry
            self.results[f"{name_prefix}_{label}"] = value

    def get_mulliken_charges(self, local_atom_index_map=None):
        self._extract_charges("Mulliken_Population_Analysis", "mulliken", local_atom_index_map=local_atom_index_map)

    def get_loewdin_charges(self, local_atom_index_map=None):
        self._extract_charges("Loewdin_Population_Analysis", "loewdin", local_atom_index_map=local_atom_index_map)

    def get_chelpg_charges(self, local_atom_index_map=None):
        self._extract_charges("CHELPG_Population_Analysis", "chelpG", preserve_case=True, local_atom_index_map=local_atom_index_map)

    def get_hirshfeld_charges(self, local_atom_index_map=None):
        self._extract_charges("Hirshfeld_Population_Analysis", "hirshfeld", local_atom_index_map=local_atom_index_map)

    def get_cm5_charges(self, extended=True, atomradius_setting="hokru", local_atom_index_map=None):
        if local_atom_index_map is None:
            local_atom_index_map = self.atom_index_map
        if not local_atom_index_map:
            print("No atoms defined for CM5.")
            return
        geom = self._last_geometry()
        if geom is None:
            print("No geometry for CM5.")
            return
        hir_raw = self._nested_get(geom, ["Hirshfeld_Population_Analysis", 0, "AtomicCharges"])
        atnos = self._nested_get(geom, ["Hirshfeld_Population_Analysis", 0, "ATNO"])
        cart = self._nested_get(geom, ["Geometry", "Coordinates", "Cartesians"])
        if hir_raw is None or atnos is None or cart is None:
            print("Missing CM5 inputs.")
            return
        try:
            hirshfeld_charges = np.array([x[0] for x in hir_raw])
            z = np.array([x[0] for x in atnos])
            xyz = np.array([[x, y, zv] for _, x, y, zv in cart]) * bohr_to_A
        except Exception as e:
            print(f"Failed CM5 prep: {e}")
            return
        atomradius = np.zeros(119)
        atomradius[:] = 0.7
        alpha = 2.474
        def _tij(i, j):
            return 0.0
        for idx, label in local_atom_index_map.items():
            i = idx - 1
            if i >= len(hirshfeld_charges):
                self.results[f"cm5_{label}"] = None
                continue
            s = 0.0
            for j in range(len(hirshfeld_charges)):
                if i == j:
                    continue
                try:
                    rij = np.linalg.norm(xyz[i] - xyz[j])
                except Exception:
                    continue
                bij = np.exp(-alpha * (rij - atomradius[z[i]] - atomradius[z[j]]))
                tij = _tij(z[i], z[j])
                s += tij * bij
            self.results[f"cm5_{label}"] = hirshfeld_charges[i] + s

    # ---------------- Chemical shift tensors ----------------
    def get_saniso(self, local_atom_index_map=None):
        if local_atom_index_map is None:
            local_atom_index_map = self.atom_index_map
        if not local_atom_index_map:
            print("Atom list is not defined. Please define it.")
            return
        geom = self._last_geometry()
        table = self._nested_get(geom, ["Chemical_Shift", 0, "saniso"]) if geom else None
        if table is None:
            print("saniso data not found; skipping.")
            return
        for at_idx, label in local_atom_index_map.items():
            value = None
            entry = self._nested_get(table, [at_idx - 1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            elif entry:
                value = entry
            self.results[f"saniso_{label}"] = value

    def get_siso(self, local_atom_index_map=None):
        if local_atom_index_map is None:
            local_atom_index_map = self.atom_index_map
        if not local_atom_index_map:
            print("Atom list is not defined. Please define it.")
            return
        geom = self._last_geometry()
        table = self._nested_get(geom, ["Chemical_Shift", 0, "siso"]) if geom else None
        if table is None:
            print("siso data not found; skipping.")
            return
        for at_idx, label in local_atom_index_map.items():
            value = None
            entry = self._nested_get(table, [at_idx - 1])
            if entry and isinstance(entry, list) and entry:
                value = entry[0]
            elif entry:
                value = entry
            self.results[f"siso_{label}"] = value

    def get_all_charges(self, local_atom_index_map=None):
        self.get_mulliken_charges(local_atom_index_map)
        self.get_loewdin_charges(local_atom_index_map)
        self.get_chelpg_charges(local_atom_index_map)
        self.get_hirshfeld_charges(local_atom_index_map)
        self.get_cm5_charges(local_atom_index_map=local_atom_index_map)
        self.get_nbo_charges(local_atom_index_map)

    # ---------------- Energetics ----------------
    def get_scfenergy(self):
        geom = self._last_geometry()
        value = self._nested_get(geom, ["DFT_Energy", "finalEn"]) if geom else None
        self.results["SCF_energy"] = value

    def get_homolumo(self):
        mos = self._nested_get(self.gbwjson, ["Molecule", "MolecularOrbitals", "MOs"])
        if mos is None:
            self.results["HOMO_energy"] = None
            self.results["LUMO_energy"] = None
            return
        energies = []
        occs = []
        for mo in mos:
            energies.append(mo.get("OrbitalEnergy"))
            occs.append(mo.get("Occupancy"))
        energies = np.array([e for e in energies if e is not None])
        occs = np.array([o for o in occs if o is not None])
        if energies.size == 0 or occs.size == 0:
            self.results["HOMO_energy"] = None
            self.results["LUMO_energy"] = None
            return
        try:
            homo_mask = (occs >= 1.0)
            lumo_mask = (occs == 0.0)
            HOMO = np.max(energies[homo_mask]) * h_to_eV if np.any(homo_mask) else None
            LUMO = np.min(energies[lumo_mask]) * h_to_eV if np.any(lumo_mask) else None
        except Exception:
            HOMO = None
            LUMO = None
        self.results["HOMO_energy"] = HOMO
        self.results["LUMO_energy"] = LUMO

    def get_enthalpy(self):
        geom = self._last_geometry()
        value = self._nested_get(geom, ["THERMOCHEMISTRY_Energies", 0, "enthalpyH"]) if geom else None
        self.results["Enthalpy"] = value

    def get_gfe(self):
        geom = self._last_geometry()
        value = self._nested_get(geom, ["THERMOCHEMISTRY_Energies", 0, "freeEnergyG"]) if geom else None
        self.results["GibbsFreeEnergy"] = value

    def _ensure_outfile_content(self):
        if self._outfile_content is None:
            try:
                self._outfile_content = self.outfile.read_text()
            except Exception as e:
                print(f"Failed to read output file '{self.outfile}': {e}")
                self._outfile_content = ""

    def _scan_job_headers(self):
        self._ensure_outfile_content()
        text = self._outfile_content
        self._job_headers = []
        if not text:
            return
        job_re = re.compile(r"(?m)^[ \t]*COMPOUND[ \t]+JOB[ \t]+(?:\{[ \t]*(\d+)[ \t]*\}|(\d+))[ \t]*$")
        for m in job_re.finditer(text):
            num = m.group(1) or m.group(2)
            try:
                num_i = int(num)
            except Exception:
                continue
            self._job_headers.append((num_i, m.start()))
        self._job_headers.sort(key=lambda x: x[1])

    def _scan_nbo_blocks(self):
        self._ensure_outfile_content()
        text = self._outfile_content
        self.nbo_sections = []
        self._nbo_blocks_meta = []
        if not text:
            return
        start_pattern = "Now starting NBO"
        end_pattern = "*** returned from  NBO  program ***"
        start_positions = []
        idx = 0
        while True:
            pos = text.find(start_pattern, idx)
            if pos == -1:
                break
            start_positions.append(pos)
            idx = pos + len(start_pattern)
        if not start_positions:
            print("No NBO start markers found in output file.")
            return
        for s in start_positions:
            e = text.find(end_pattern, s)
            if e == -1:
                print(f"Warning: NBO start at {s} without matching end marker; skipping.")
                continue
            block = text[s:e + len(end_pattern)]
            self.nbo_sections.append(block)
            self._nbo_blocks_meta.append((s, e + len(end_pattern)))
        if not self.nbo_sections:
            print("No complete NBO blocks detected.")

    def _select_nbo_block_by_job(self, job_number:int):
        if not self._job_headers:
            self._scan_job_headers()
        if not self._job_headers:
            print("No COMPOUND JOB headers found; falling back to first NBO block.")
            return 0 if self.nbo_sections else None
        header_pos = None
        next_header_pos = None
        for i, (num, pos) in enumerate(self._job_headers):
            if num == job_number:
                header_pos = pos
                if i + 1 < len(self._job_headers):
                    next_header_pos = self._job_headers[i+1][1]
                break
        if header_pos is None:
            print(f"COMPOUND JOB {job_number} not found; falling back to first NBO block.")
            return 0 if self.nbo_sections else None
        for idx, (s, e) in enumerate(self._nbo_blocks_meta):
            if s >= header_pos and (next_header_pos is None or s < next_header_pos):
                return idx
        print(f"No NBO block found after COMPOUND JOB {job_number}; falling back to first NBO block.")
        return 0 if self.nbo_sections else None

    def init_nbo(self, job_number:int|None=None):
        self._clear_nbo_data()
        if not self.nbo_sections or not self._nbo_blocks_meta:
            self._scan_nbo_blocks()
        selected_index = None
        effective_job = job_number if job_number is not None else self.nbo_job_number
        if effective_job is not None:
            selected_index = self._select_nbo_block_by_job(effective_job)
        if selected_index is None:
            if not self.nbo_sections:
                self.nbo_section = None
                return
            if not (0 <= self.nbo_block_index < len(self.nbo_sections)):
                print(f"Block index {self.nbo_block_index} out of range; defaulting to 0.")
                self.nbo_block_index = 0
            selected_index = self.nbo_block_index
        self.nbo_block_index = selected_index
        self.nbo_section = self.nbo_sections[selected_index]
        self.nboresults = {
            'source_file': str(self.outfile),
            'nbo_block_index': self.nbo_block_index,
            'job_number': effective_job,
            'analysis_type': 'NBO 7.0',
            'natural_population_analysis': {'atoms': {}, 'totals': {}},
            'natural_atomic_orbitals': {},
            'natural_bond_orbitals': {},
            'bond_orbital_details': {},
            'nho_directionality': {},
            'second_order_perturbation': {},
            'analysis_info': {}
        }

    def get_NPA(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        npa_start = self.nbo_section.find("Summary of Natural Population Analysis:")
        npa_end = self.nbo_section.find("Natural Electron Configuration", npa_start)
        if npa_end == -1:
            npa_end = self.nbo_section.find("\n\n NATURAL BOND ORBITAL ANALYSIS:", npa_start)
        if npa_start != -1 and npa_end != -1:
            npa_section = self.nbo_section[npa_start:npa_end]
            atom_properties = []
            atom_lines = re.findall(r'\n\s+([A-Z][a-z]?)\s+(\d+)\s+([-\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', npa_section)
            for atom_data in atom_lines:
                atom_symbol = atom_data[0]
                atom_num = int(atom_data[1])
                while len(atom_properties) <= atom_num:
                    atom_properties.append(None)
                atom_properties[atom_num] = {
                    'atom': atom_symbol,
                    'charge': float(atom_data[2]),
                    'core': float(atom_data[3]),
                    'valence': float(atom_data[4]),
                    'rydberg': float(atom_data[5]),
                    'total': float(atom_data[6])
                }
            self.nboresults['natural_population_analysis']['atoms'] = {i: prop for i, prop in enumerate(atom_properties) if prop is not None}
            total_match = re.search(r'\*\s*Total\s*\*\s+([-\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)', npa_section)
            if total_match:
                total_properties = {
                    'charge': float(total_match.group(1)),
                    'core': float(total_match.group(2)),
                    'valence': float(total_match.group(3)),
                    'rydberg': float(total_match.group(4)),
                    'total': float(total_match.group(5))
                }
                self.nboresults['natural_population_analysis']['totals'] = total_properties
        else:
            print("Could not find Natural Population Analysis section")

    def get_NAO(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        nao_start = self.nbo_section.find("NATURAL POPULATIONS:  Natural atomic orbital occupancies")
        nao_end = self.nbo_section.find("Summary of Natural Population Analysis:", nao_start)
        nao_properties = []
        if nao_start != -1 and nao_end != -1:
            nao_subsection = self.nbo_section[nao_start:nao_end]
            nao_lines = re.findall(r'\n\s+(\d+)\s+([A-Z][a-z]?)\s+(\d+)\s+(\S+)\s+(\S+\([^)]+\))\s+([\d.]+)\s+([-\d.]+)', nao_subsection)
            for line_data in nao_lines:
                orbital_type = line_data[4].split('(')[0]
                orbital = line_data[4].split('(')[1].strip(')').strip()
                nao_properties.append({
                    'orbital_num': int(line_data[0]),
                    'atom': line_data[1],
                    'atom_num': int(line_data[2]),
                    'angular': line_data[3],
                    'orbital_type': orbital_type,
                    'orbital': orbital,
                    'occupancy': float(line_data[5]),
                    'energy': float(line_data[6])
                })
            self.nboresults['natural_atomic_orbitals'] = nao_properties
        else:
            print("Could not find NAO section")

    def get_NBO_summary(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        nbo_summary_start = self.nbo_section.find("NATURAL BOND ORBITALS (Summary):")
        nbo_summary_end = self.nbo_section.find("$CHOOSE", nbo_summary_start)
        if nbo_summary_end == -1:
            nbo_summary_end = self.nbo_section.find("Checkpoint orbital", nbo_summary_start)
        nbo_properties = []
        if nbo_summary_start != -1 and nbo_summary_end != -1:
            nbo_summary_section = self.nbo_section[nbo_summary_start:nbo_summary_end]
            nbo_lines = re.findall(r'\n\s+(\d+)\.\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+([A-Z][a-z]?\s+\d+(?:\s*-\s*[A-Z][a-z]?\s+\d+)*)\s+([\d.]+)\s+([-\d.]+)', nbo_summary_section)
            for line_data in nbo_lines:
                atoms_string = line_data[3].strip()
                atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_string)]
                nbo_properties.append({
                    'nbo_num': int(line_data[0]),
                    'nbo_type': line_data[1],
                    'bond_index': int(line_data[2]),
                    'atoms': atom_numbers,
                    'occupancy': float(line_data[4]),
                    'energy': float(line_data[5])
                })
            self.nboresults['natural_bond_orbitals'] = nbo_properties
        else:
            print("Could not find NBO summary section")

    def get_orbital_details(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        bond_detail_start = self.nbo_section.find("(Occupancy)   Bond orbital / Coefficients / Hybrids")
        lewis_start = self.nbo_section.find("---------------- Lewis", bond_detail_start)
        non_lewis_start = self.nbo_section.find("---------------- non-Lewis", lewis_start)
        bond_orbital_details = []
        if bond_detail_start != -1 and lewis_start != -1:
            if non_lewis_start != -1:
                end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
                detail_end = -1
                for marker in end_markers:
                    pos = self.nbo_section.find(marker, non_lewis_start)
                    if pos != -1:
                        detail_end = pos; break
            else:
                detail_end = -1
                end_markers = ["NHO DIRECTIONALITY", "SECOND ORDER", "\n\n NATURAL BOND ORBITALS (Summary)"]
                for marker in end_markers:
                    pos = self.nbo_section.find(marker, lewis_start)
                    if pos != -1:
                        detail_end = pos; break
            if detail_end != -1:
                bond_detail_section = self.nbo_section[lewis_start:detail_end]
                bond_entries = re.finditer(r'\n\s+(\d+)\.\s+\(([\d.]+)\)\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)(?=\n\s+\d+\.\s+\(|\Z)', bond_detail_section, re.DOTALL)
                for bond_entry in bond_entries:
                    bond_num = int(bond_entry.group(1))
                    occupancy = float(bond_entry.group(2))
                    bond_type = str(bond_entry.group(3))
                    bond_index = int(bond_entry.group(4))
                    atoms_and_contrib = bond_entry.group(5)
                    atoms_line_match = re.match(r'(.*?)(?=\n|\()', atoms_and_contrib)
                    atoms_line = atoms_line_match.group(1).strip() if atoms_line_match else ""
                    atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_line)]
                    bond_info = {
                        'bond_num': bond_num,
                        'occupancy': occupancy,
                        'bond_type': bond_type,
                        'bond_index': bond_index,
                        'atoms': atom_numbers,
                        'contributions': []
                    }
                    contrib_pattern = r'\(\s*([\d.]+)%\)\s+([-\d.]+)\*\s+([A-Z][a-z]?)\s+(\d+)\s+s\(\s*([\d.]+)%\)p\s*([\d.]+)\(\s*([\d.]+)%\)'
                    contributions = re.finditer(contrib_pattern, atoms_and_contrib)
                    for contrib in contributions:
                        bond_info['contributions'].append({
                            'percentage': float(contrib.group(1)),
                            'coefficient': float(contrib.group(2)),
                            'atom': str(contrib.group(3)),
                            'atom_num': int(contrib.group(4)),
                            's_percent': float(contrib.group(5)),
                            'p_power': float(contrib.group(6)),
                            'p_percent': float(contrib.group(7))
                        })
                    bond_orbital_details.append(bond_info)
                self.nboresults['bond_orbital_details'] = bond_orbital_details
            else:
                print("Could not determine end of bond detail section")
        else:
            print("Could not find bond orbital detail section")

    def get_NHO_directionality(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        dir_start = self.nbo_section.find("NHO DIRECTIONALITY AND BOND BENDING")
        dir_end = self.nbo_section.find("\n SECOND ORDER PERTURBATION", dir_start)
        if dir_end == -1: dir_end = self.nbo_section.find("\n\n NATURAL BOND ORBITALS", dir_start)
        nbo_directionality = []
        if dir_start != -1 and dir_end != -1:
            dir_section = self.nbo_section[dir_start:dir_end]
            if "None exceeding thresholds" not in dir_section:
                dir_lines = re.findall(r'\n\s+(\d+)\.\s+(BD|LP|CR)\s*\(\s*(\d+)\)\s+([A-Z][a-z]?\s+\d+(?:\s*-\s*[A-Z][a-z]?\s+\d+)*)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)\s+([\d.-]+)', dir_section)
                for line_data in dir_lines:
                    atoms_string = line_data[3].strip()
                    atom_numbers = [int(m[1]) for m in re.findall(r'([A-Z][a-z]?)\s+(\d+)', atoms_string)]
                    nbo_directionality.append({
                        'nbo_num': int(line_data[0]),
                        'nbo_type': line_data[1],
                        'bond_index': int(line_data[2]),
                        'atoms': atom_numbers,
                        'line_theta': float(line_data[4]) if line_data[4] != '--' else None,
                        'line_phi': float(line_data[5]) if line_data[5] != '--' else None,
                        'hybrid_theta': float(line_data[6]) if line_data[6] != '--' else None,
                        'hybrid_phi': float(line_data[7]) if line_data[7] != '--' else None,
                        'deviation': float(line_data[8]) if line_data[8] != '--' else None
                    })
            self.nboresults['nho_directionality'] = nbo_directionality
        else:
            print("Could not find NHO Directionality section")

    def get_second_order_perturbation(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        second_start = self.nbo_section.find("SECOND ORDER PERTURBATION THEORY ANALYSIS")
        second_end = self.nbo_section.find("\n\n NATURAL BOND ORBITALS (Summary):", second_start)
        if second_end == -1: second_end = self.nbo_section.find("\n $CHOOSE", second_start)
        second_order_perturbation = []
        if second_start != -1 and second_end != -1:
            second_section = self.nbo_section[second_start:second_end]
            if "None above threshold" not in second_section and "within unit" in second_section:
                interaction_pattern = r'\n\s+(BD\*?|LP|CR|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+->\s+(BD\*?|LP\*?|CR\*?|RY)\s*\(\s*(\d+)\)\s+(.*?)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
                interactions = re.finditer(interaction_pattern, second_section)
                for inter in interactions:
                    second_order_perturbation.append({
                        'donor_type': inter.group(1),
                        'donor_index': int(inter.group(2)),
                        'donor_atoms': inter.group(3).strip(),
                        'acceptor_type': inter.group(4),
                        'acceptor_index': int(inter.group(5)),
                        'acceptor_atoms': inter.group(6).strip(),
                        'e2_energy': float(inter.group(7)),
                        'energy_diff': float(inter.group(8)),
                        'fock_element': float(inter.group(9))
                    })
            self.nboresults['second_order_perturbation'] = second_order_perturbation
        else:
            print("Could not find Second-Order Perturbation section")

    def get_cycle_info(self):
        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        cycle_start = self.nbo_section.find("NATURAL BOND ORBITAL ANALYSIS:")
        cycle_end = self.nbo_section.find("Structure accepted:", cycle_start)
        if cycle_end == -1:
            cycle_end = self.nbo_section.find("\n\n", cycle_start + 100)
        nbo_analysis_info = {}
        if cycle_start != -1 and cycle_end != -1:
            cycle_section = self.nbo_section[cycle_start:cycle_end + 200]
            cycle_match = re.search(r'\n\s+(\d+)\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)', cycle_section)
            if cycle_match:
                nbo_analysis_info = {
                    'cycle': int(cycle_match.group(1)),
                    'center': int(cycle_match.group(2)),
                    'threshold': float(cycle_match.group(3)),
                    'lewis_occupancy': float(cycle_match.group(4)),
                    'non_lewis_occupancy': float(cycle_match.group(5)),
                    'num_core': int(cycle_match.group(6)),
                    'num_bonds': int(cycle_match.group(7)),
                    'num_non_bonding_cores': int(cycle_match.group(8)),
                    'num_lone_pairs': int(cycle_match.group(9)),
                    'low_occ_lewis': int(cycle_match.group(10)),
                    'high_occ_non_lewis': int(cycle_match.group(11))
                }
                self.nboresults['analysis_info'] = nbo_analysis_info
        else:
            print("Could not find NBO analysis section")

    def get_nbo_charges(self, local_atom_index_map=None):
        if not self.atom_index_map or not self.atom_indices:
            print("Atom list is not defined. Please define it.")
            return
        if not local_atom_index_map:
            local_atom_index_map = self.atom_index_map

        if self.nbo_section is None:
            self.init_nbo()
        if self.nbo_section is None:
            return
        if not self.nboresults.get('natural_population_analysis', {}).get('atoms'):
            self.get_NPA()
        atoms_dict = self.nboresults.get('natural_population_analysis', {}).get('atoms', {})

        for at_idx in local_atom_index_map.keys():
            npa_data = atoms_dict.get(at_idx)
            self.results[f"nbo_{local_atom_index_map[at_idx]}"] = npa_data['charge'] if npa_data else None

    # ---------------- Steric descriptors ----------------
    def compute_sterimol_descriptors(self, xyzfile, bond_list, label):
        xyzfile = str(xyzfile)
        mol = db.dbstep(xyzfile, atom1=bond_list[0], atom2=bond_list[1], commandline=True, verbose=False, sterimol=True, quiet=True)
        self.results[f'Sterimol_{label}_Bmin'] = mol.Bmin
        self.results[f'Sterimol_{label}_Bmax'] = mol.Bmax
        self.results[f'Sterimol_{label}_L'] = float(mol.L)

    def compute_buried_volume(self, xyzfile, center_atom=None, label=None, radius=3.5):
        if not center_atom:
            if not self.atom_index_map:
                print("No atoms defined for buried volume calculation.")
                return
            # calculate for each atom in atom_index_map
            for idx, atom_label in self.atom_index_map.items():
                self.compute_buried_volume(xyzfile, center_atom=idx, label=atom_label, radius=radius)
            return
        
        xyzfile = str(xyzfile)
        mol = db.dbstep(xyzfile, atom1=center_atom, commandline=True, volume=True, r=radius, verbose=False, quiet=True, vshell=0.5)
        self.results[f'BuriedVolume_{radius}A_{label}'] = float(mol.bur_vol)
        self.results[f'BuriedShell_{radius}A_{label}'] = float(mol.bur_shell)
    
    # ---------------- Custom result helpers ----------------
    def add_custom_result(self, key, value, prefix=""):
        self.results[prefix+"_"+key] = value

    def add_custom_results(self, results_dict, prefix=""):
        for k, v in results_dict.items():
            self.results[prefix+"_"+k] = v

    # ---------------- Output ----------------
    def write_results_to_file(self, output_path=None):
        if output_path is None:
            output_path = self.directory / f"{self.basename}_descriptors.json"
        try:
            with open(output_path, 'w') as f:
                json.dump(self.results, f, indent=4)
        except Exception as e:
            print(f"Failed writing results file: {e}")

class RedoxSet:
    """Aggregate redox descriptors using unified atom index mapping from OrcaReader."""
    def __init__(self, neutralReader=None, cationReader=None, anionReader=None, descriptorPrefixes=None):
        self.neutralReader = neutralReader
        self.cationReader = cationReader
        self.anionReader = anionReader
        self.descriptorPrefixes = descriptorPrefixes or ["mulliken", "loewdin", "cm5", "chelpg", "hirshfeld"]
        self.results = {}

    @property
    def atom_indices(self):
        return self.neutralReader.atom_indices if self.neutralReader else []

    def _atom_key(self, prefix, atom_index):
        if self.neutralReader:
            return self.neutralReader._atom_key(prefix, atom_index)
        return f"{prefix}_{atom_index}"

    def _pull(self, reader, prefix, idx):
        if reader is None:
            return None
        return reader.results.get(self._atom_key(prefix, idx))

    def get_redox_properties(self):
        n = self.neutralReader.results.get('SCF_energy') if self.neutralReader else None
        c = self.cationReader.results.get('SCF_energy') if self.cationReader else None
        a = self.anionReader.results.get('SCF_energy') if self.anionReader else None
        self.results['Vertical_Ionization_Potential'] = (n - c) if (n is not None and c is not None) else None
        self.results['Vertical_Electron_Affinity'] = (a - n) if (a is not None and n is not None) else None
        ip = self.results['Vertical_Ionization_Potential']
        ea = self.results['Vertical_Electron_Affinity']
        if ip is not None and ea is not None:
            hardness = (ip - ea) / 2
            electronegativity = (ip + ea) / 2
            self.results['Hardness'] = hardness
            self.results['Electronegativity'] = electronegativity
            self.results['Electrophilicity'] = (electronegativity ** 2) / (2 * hardness) if hardness else None
        else:
            self.results['Hardness'] = None
            self.results['Electronegativity'] = None
            self.results['Electrophilicity'] = None

    def get_fukui_functions(self):
        if not self.atom_indices:
            print("No atom indices for Fukui functions.")
            return
        for prefix in self.descriptorPrefixes:
            for idx in self.atom_indices:
                qN = self._pull(self.neutralReader, prefix, idx)
                qC = self._pull(self.cationReader, prefix, idx)
                qA = self._pull(self.anionReader, prefix, idx)
                f_plus = (qN - qA) if (qN is not None and qA is not None) else None
                f_minus = (qC - qN) if (qC is not None and qN is not None) else None
                f_zero = (0.5 * (f_plus + f_minus)) if (f_plus is not None and f_minus is not None) else None
                self.results[self._atom_key(f"fplus_{prefix}", idx)] = f_plus
                self.results[self._atom_key(f"fminus_{prefix}", idx)] = f_minus
                self.results[self._atom_key(f"fzero_{prefix}", idx)] = f_zero

    def aggregate(self):
        self.get_redox_properties()
        self.get_fukui_functions()
        return self.results

In [96]:
# Test usage of new OrcaReader / RedoxSet atom index handling
reader = OrcaReader(directory="results/N_CC1_CC_CN_N1", basename="singlepoint", out_file_path="results/N_CC1_CC_CN_N1/calcParam.out")
reader.set_atoms({1:"N1",2:"N2",3:"C3",4:"C4",5:"C5",6:"C6"})
reader.parse_property_json()
reader.parse_gbw_json()
reader.get_scfenergy()
reader.get_homolumo()
reader.get_all_charges()
print("Atom map:", reader.atom_index_map)
print("Sample Mulliken keys:", {k:v for k,v in reader.results.items() if k.startswith("mulliken_")})

Atom map: {1: 'N1', 2: 'N2', 3: 'C3', 4: 'C4', 5: 'C5', 6: 'C6'}
Sample Mulliken keys: {'mulliken_N1': -0.2510661962970566, 'mulliken_N2': -0.10205795109428806, 'mulliken_C3': 0.5136274212653493, 'mulliken_C4': -0.13701241555566845, 'mulliken_C5': -0.01746944711022369, 'mulliken_C6': 0.05852428276045174}


In [70]:
def get_substituent_weight(mol, ring_atom_idx, substituent_root_idx):
    """
    Calculates the molecular weight of a substituent using RDKit's
    fragmentation tools.

    Args:
        mol (rdkit.Chem.Mol): The molecule object.
        ring_atom_idx (int): The integer index of the atom in the pyridazine ring.
        substituent_root_idx (int): The integer index of the substituent atom attached to the ring.

    Returns:
        float: The exact molecular weight of the substituent fragment, or 0.0 if not found.
    """
    # 1. Find the bond to break
    bond_to_break = mol.GetBondBetweenAtoms(ring_atom_idx, substituent_root_idx)
    if not bond_to_break:
        return 0.0

    # 2. Break the molecule at that bond, creating dummy atoms
    frags_mol = rdmolops.FragmentOnBonds(mol, [bond_to_break.GetIdx()], addDummies=True)

    # 3. Get the resulting disconnected fragments as separate molecule objects
    sub_mols = Chem.GetMolFrags(frags_mol, asMols=True)

    # 4. Identify the correct fragment and calculate its weight
    for sub_mol in sub_mols:
        for atom in sub_mol.GetAtoms():
            if atom.GetAtomicNum() == 0:  # This is a dummy atom
                if atom.HasProp('_fraggedOnAtom') and atom.GetIntProp('_fraggedOnAtom') == ring_atom_idx:
                    return Descriptors.ExactMolWt(sub_mol)

    return 0.0


def get_canonical_pyridazine_indices(mol, match_indices):
    """
    Reorders pyridazine indices based on substituent weight to give the
    heaviest substituents the lowest locants.
    Returns 0-based indices suitable for RDKit internal use.
    """
    ring_indices_set = set(match_indices)

    # Generate the two possible ring paths (logic is unchanged)
    neighbors = {idx: [n.GetIdx() for n in mol.GetAtomWithIdx(idx).GetNeighbors() if n.GetIdx() in ring_indices_set] for idx in match_indices}
    n_indices = [idx for idx in match_indices if mol.GetAtomWithIdx(idx).GetAtomicNum() == 7]
    if len(n_indices) != 2:
        return None
    n1_cand, n2_cand = n_indices[0], n_indices[1]

    paths = []
    for start_node, second_node in [(n1_cand, n2_cand), (n2_cand, n1_cand)]:
        if second_node not in neighbors[start_node]:
            continue
        path = [start_node, second_node]
        prev_node, curr_node = path[-2], path[-1]
        while len(path) < 6:
            next_node = [n for n in neighbors[curr_node] if n != prev_node][0]
            path.append(next_node)
            prev_node, curr_node = curr_node, next_node
        paths.append(tuple(path))

    if len(paths) < 2:
        return min(paths) if paths else None

    path_A, path_B = paths[0], paths[1]

    # Analyze substituents for each path
    path_scores = []
    for path in [path_A, path_B]:
        substituents = []
        for i in range(2, 6):  # Iterate through carbons C3-C6
            locant = i + 1
            atom_idx = path[i]
            atom = mol.GetAtomWithIdx(atom_idx)
            for neighbor in atom.GetNeighbors():
                if neighbor.GetIdx() not in ring_indices_set:
                    weight = get_substituent_weight(mol, atom_idx, neighbor.GetIdx())
                    substituents.append({'locant': locant, 'weight': weight})
        substituents.sort(key=lambda x: x['weight'], reverse=True)
        path_scores.append([sub['locant'] for sub in substituents])

    score_A, score_B = path_scores

    # Compare scores and return the winning path (logic is unchanged)
    if score_A < score_B:
        return path_A
    elif score_B < score_A:
        return path_B
    else:
        return min(path_A, path_B)


def get_canonical_pyridazine_rings(xyz_file, charge=0):
    """
    Identifies pyridazine rings in an XYZ file and returns their atoms in a
    standardized N1, N2, C3, C4, C5, C6 order.

    Indices returned are now 1-based (human-friendly). If you need 0-based
    indices for RDKit operations, subtract 1 when using them.

    Args:
        xyz_file (str | Path | Mol): The path to the XYZ file or an RDKit Mol.
        charge (int, optional): The charge of the molecule. Defaults to 0.

    Returns:
        list[list[int]]: A list of canonical 1-based index lists for all pyridazine rings.
    """
    if isinstance(xyz_file, Path):
        xyz_file = str(xyz_file)
    if isinstance(xyz_file, str):
        mol = Chem.MolFromXYZFile(xyz_file)
        if mol is None:
            print(f"Error: Could not read molecule from {xyz_file}")
            return []
    elif isinstance(xyz_file, Chem.Mol):
        mol = xyz_file
    else:
        print("Error: Invalid input type.")
        return []

    if mol.GetNumBonds() == 0:
        rdDetermineBonds.DetermineBonds(mol, charge)

    pyridazine_smarts = "c1nnccc1"
    pyridazine_pattern = Chem.MolFromSmarts(pyridazine_smarts)
    matches = mol.GetSubstructMatches(pyridazine_pattern)

    if not matches:
        print("No pyridazine rings found.")
        return []

    canonical_rings = []
    for match in matches:
        canonical_indices_0based = get_canonical_pyridazine_indices(mol, match)
        if canonical_indices_0based:
            # Convert to 1-based indices here
            canonical_rings.append([i + 1 for i in canonical_indices_0based])

    return canonical_rings

In [71]:
def get_connected_atom(xyz_file, ring_atoms, atom_idx):
    """
    Given an XYZ file (path or RDKit Mol) and a set of *1-based* ring atom
    indices, find the atom connected to the specified *1-based* atom index
    that is not part of the ring.

    Inputs are now expected to be 1-based (human-friendly). The function
    converts them internally to 0-based RDKit indices. The returned neighbor
    index is also 1-based. If no non-ring neighbor is found, returns None.

    Args:
        xyz_file (str | Path | rdkit.Chem.Mol): Path to XYZ file or RDKit Mol.
        ring_atoms (list[int]): 1-based indices of ring atoms.
        atom_idx (int): 1-based index of the atom whose external neighbor is sought.

    Returns:
        int | None: 1-based index of connected atom not in the ring, or None.
    """
    if isinstance(xyz_file, Path):
        xyz_file = str(xyz_file)
    if isinstance(xyz_file, str):
        mol = Chem.MolFromXYZFile(xyz_file)
        if mol is None:
            print(f"Error: Could not read molecule from {xyz_file}")
            return None
    elif isinstance(xyz_file, Chem.Mol):
        mol = xyz_file
    else:
        print("Error: Invalid input type.")
        return None

    if mol.GetNumBonds() == 0:
        rdDetermineBonds.DetermineBonds(mol)

    # Validate atom_idx
    if atom_idx < 1 or atom_idx > mol.GetNumAtoms():
        print("Error: atom_idx out of bounds for 1-based indexing.")
        return None

    # Convert inputs to 0-based
    ring_atom_set = set(i - 1 for i in ring_atoms)
    atom0 = atom_idx - 1

    atom = mol.GetAtomWithIdx(atom0)
    for neighbor in atom.GetNeighbors():
        neighbor_idx0 = neighbor.GetIdx()
        if neighbor_idx0 not in ring_atom_set:
            return neighbor_idx0 + 1  # convert back to 1-based

    return None

In [149]:
singlepoint = OrcaReader(directory="results/N_CC1_CC_CN_N1",basename="singlepoint", out_file_path="results/N_CC1_CC_CN_N1/calcParam.out")
ring_atoms = get_canonical_pyridazine_rings("results/N_CC1_CC_CN_N1/optimization.xyz")[0]
atom_labels = ["N1", "N2", "C3", "C4", "C5", "C6"]
singlepoint.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})

singlepoint.parse_property_json()
singlepoint.set_nbo_job_number(4)
singlepoint.get_all_charges()
singlepoint.parse_gbw_json()
singlepoint.get_homolumo()
singlepoint.get_scfenergy()

singlepoint.compute_sterimol_descriptors("results/N_CC1_CC_CN_N1/optimization.xyz", [ring_atoms[2], get_connected_atom("results/N_CC1_CC_CN_N1/optimization.xyz", ring_atoms, ring_atoms[2])], "C3")
singlepoint.compute_sterimol_descriptors("results/N_CC1_CC_CN_N1/optimization.xyz", [ring_atoms[5], get_connected_atom("results/N_CC1_CC_CN_N1/optimization.xyz", ring_atoms, ring_atoms[5])], "C6")
singlepoint.compute_buried_volume("results/N_CC1_CC_CN_N1/optimization.xyz", radius=3.5)

nmr_reader = OrcaReader(directory="results/N_CC1_CC_CN_N1", basename="nmr")
nmr_reader.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
nmr_reader.parse_property_json()
nmr_reader.get_siso()
nmr_reader.get_saniso()

nmr_reader_nics = OrcaReader(directory="nics_results/N_CC1_CC_CN_N1_nics", basename="N_CC1_CC_CN_N1_nics")
nmr_reader_nics.set_atoms({1 : "NICS0", 2 : "NICS1", 3 : "NICS-1"})
nmr_reader_nics.parse_property_json()
nmr_reader_nics.get_siso()
nmr_reader_nics.get_saniso()

thermo = OrcaReader(directory="results/N_CC1_CC_CN_N1",basename="frequency", out_file_path="results/N_CC1_CC_CN_N1/calcParam.out")
thermo.parse_property_json()
thermo.get_enthalpy()
thermo.get_gfe()

In [146]:
protonated0 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated0",basename="singlepoint", out_file_path="protonated/N_CC1_CC_CN_N1_protonated0/calcParam.out")
protonated0.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
protonated0.parse_property_json()
protonated0.parse_gbw_json()
protonated0.set_nbo_job_number(4)
protonated0.get_homolumo()
protonated0.get_scfenergy()
protonated0.get_all_charges()

protonated1 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated1",basename="singlepoint", out_file_path="protonated/N_CC1_CC_CN_N1_protonated1/calcParam.out")
protonated1.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
protonated1.parse_property_json()
protonated1.parse_gbw_json()
protonated1.set_nbo_job_number(4)
protonated1.get_homolumo()
protonated1.get_scfenergy()
protonated1.get_all_charges()

protonated0_nmr = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated0",basename="nmr", out_file_path="protonated/N_CC1_CC_CN_N0_protonated0/calcParam.out")
protonated0_nmr.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
protonated0_nmr.parse_property_json()
protonated0_nmr.get_siso()
protonated0_nmr.get_saniso()

protonated1_nmr = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated1",basename="nmr", out_file_path="protonated/N_CC1_CC_CN_N0_protonated1/calcParam.out")
protonated1_nmr.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
protonated1_nmr.parse_property_json()
protonated1_nmr.get_siso()
protonated1_nmr.get_saniso()

protonated0_thermo = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated0",basename="frequency", out_file_path="protonated/N_CC1_CC_CN_N1_protonated0/calcParam.out")
protonated0_thermo.parse_property_json()
protonated0_thermo.get_enthalpy()
protonated0_thermo.get_gfe()

protonated1_thermo = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated1",basename="frequency", out_file_path="protonated/N_CC1_CC_CN_N1_protonated1/calcParam.out")
protonated1_thermo.parse_property_json()
protonated1_thermo.get_enthalpy()
protonated1_thermo.get_gfe()

nmr_reader_nics_protonated0 = OrcaReader(directory="nics_results/N_CC1_CC_CN_N1_protonated0_nics", basename="N_CC1_CC_CN_N1_protonated0_nics")
nmr_reader_nics_protonated0.set_atoms({1 : "NICS0_p0", 2 : "NICS1_p0", 3 : "NICS-1_p0"})
nmr_reader_nics_protonated0.parse_property_json()
nmr_reader_nics_protonated0.get_siso()
nmr_reader_nics_protonated0.get_saniso()

nmr_reader_nics_protonated1 = OrcaReader(directory="nics_results/N_CC1_CC_CN_N1_protonated1_nics", basename="N_CC1_CC_CN_N1_protonated1_nics")
nmr_reader_nics_protonated1.set_atoms({1 : "NICS0_p1", 2 : "NICS1_p1", 3 : "NICS-1_p1"})
nmr_reader_nics_protonated1.parse_property_json()
nmr_reader_nics_protonated1.get_siso()
nmr_reader_nics_protonated1.get_saniso()


In [153]:
reduced = OrcaReader(directory="results/N_CC1_CC_CN_N1",basename="reduction", out_file_path="results/N_CC1_CC_CN_N1/calcParam.out")
reduced.parse_property_json()
reduced.parse_gbw_json()
reduced.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
reduced.set_nbo_job_number(7)
reduced.get_scfenergy()
reduced.get_all_charges()

oxidized = OrcaReader(directory="results/N_CC1_CC_CN_N1",basename="oxidation", out_file_path="results/N_CC1_CC_CN_N1/calcParam.out")
oxidized.parse_property_json()
oxidized.parse_gbw_json()
oxidized.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
oxidized.set_nbo_job_number(6)
oxidized.get_scfenergy()
oxidized.get_all_charges()

redox = RedoxSet(neutralReader=singlepoint, cationReader=oxidized, anionReader=reduced)
redox.aggregate()

{'Vertical_Ionization_Potential': -0.2808908459347208,
 'Vertical_Electron_Affinity': -0.08543864123373623,
 'Hardness': -0.09772610235049228,
 'Electronegativity': -0.18316474358422852,
 'Electrophilicity': -0.17164975623376624,
 'fplus_mulliken_N1': 0.06799208224281905,
 'fminus_mulliken_N1': 0.25667545615046183,
 'fzero_mulliken_N1': 0.16233376919664044,
 'fplus_mulliken_N2': 0.18250405111777557,
 'fminus_mulliken_N2': 0.19435143318511283,
 'fzero_mulliken_N2': 0.1884277421514442,
 'fplus_mulliken_C3': 0.1402105667168536,
 'fminus_mulliken_C3': 0.1089045837440521,
 'fzero_mulliken_C3': 0.12455757523045285,
 'fplus_mulliken_C4': 0.028849819096832263,
 'fminus_mulliken_C4': 0.05128792739276555,
 'fzero_mulliken_C4': 0.04006887324479891,
 'fplus_mulliken_C5': 0.16087150615634815,
 'fminus_mulliken_C5': 0.0635778958729416,
 'fzero_mulliken_C5': 0.11222470101464488,
 'fplus_mulliken_C6': 0.09005673538048331,
 'fminus_mulliken_C6': 0.09829326592037457,
 'fzero_mulliken_C6': 0.094175000650

In [156]:
reduced_protonated0 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated0",basename="reduction", out_file_path="protonated/N_CC1_CC_CN_N1_protonated0/calcParam.out")
reduced_protonated0.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
reduced_protonated0.parse_property_json()
reduced_protonated0.parse_gbw_json()
reduced_protonated0.set_nbo_job_number(7)
reduced_protonated0.get_homolumo()
reduced_protonated0.get_scfenergy()
reduced_protonated0.get_all_charges()

reduced_protonated1 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated1",basename="reduction", out_file_path="protonated/N_CC1_CC_CN_N1_protonated1/calcParam.out")
reduced_protonated1.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
reduced_protonated1.parse_property_json()
reduced_protonated1.parse_gbw_json()
reduced_protonated1.set_nbo_job_number(7)
reduced_protonated1.get_homolumo()
reduced_protonated1.get_scfenergy()
reduced_protonated1.get_all_charges()

oxidized_protonated0 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated0",basename="oxidation", out_file_path="protonated/N_CC1_CC_CN_N1_protonated0/calcParam.out")
oxidized_protonated0.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
oxidized_protonated0.parse_property_json()
oxidized_protonated0.parse_gbw_json()
oxidized_protonated0.set_nbo_job_number(6)
oxidized_protonated0.get_homolumo()
oxidized_protonated0.get_scfenergy()
oxidized_protonated0.get_all_charges()

oxidized_protonated1 = OrcaReader(directory="protonated/N_CC1_CC_CN_N1_protonated1",basename="oxidation", out_file_path="protonated/N_CC1_CC_CN_N1_protonated1/calcParam.out")
oxidized_protonated1.set_atoms({idx: label for idx, label in zip(ring_atoms, atom_labels)})
oxidized_protonated1.parse_property_json()
oxidized_protonated1.parse_gbw_json()
oxidized_protonated1.set_nbo_job_number(6)
oxidized_protonated1.get_homolumo()
oxidized_protonated1.get_scfenergy()
oxidized_protonated1.get_all_charges()

redox_protonated0 = RedoxSet(neutralReader=protonated0, cationReader=oxidized_protonated0, anionReader=reduced_protonated0)
redox_protonated0.aggregate()
redox_protonated1 = RedoxSet(neutralReader=protonated1, cationReader=oxidized_protonated1, anionReader=reduced_protonated1)
redox_protonated1.aggregate()

{'Vertical_Ionization_Potential': -0.36587672227739176,
 'Vertical_Electron_Affinity': -0.14738445244444165,
 'Hardness': -0.10924613491647506,
 'Electronegativity': -0.2566305873609167,
 'Electrophilicity': -0.30142603406318347,
 'fplus_mulliken_N1': 0.1178000779398527,
 'fminus_mulliken_N1': 0.16375702348072174,
 'fzero_mulliken_N1': 0.14077855071028722,
 'fplus_mulliken_N2': 0.140900004773588,
 'fminus_mulliken_N2': 0.05013150595229643,
 'fzero_mulliken_N2': 0.09551575536294221,
 'fplus_mulliken_C3': 0.13877504520344264,
 'fminus_mulliken_C3': 0.09503026280742599,
 'fzero_mulliken_C3': 0.11690265400543431,
 'fplus_mulliken_C4': 0.03504041569002769,
 'fminus_mulliken_C4': 0.10329754632297217,
 'fzero_mulliken_C4': 0.06916898100649993,
 'fplus_mulliken_C5': 0.1768002380343079,
 'fminus_mulliken_C5': 0.05042905379976492,
 'fzero_mulliken_C5': 0.1136146459170364,
 'fplus_mulliken_C6': 0.06129876084426389,
 'fminus_mulliken_C6': 0.10473964961581927,
 'fzero_mulliken_C6': 0.08301920523004